In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("""
    CREATE TABLE IF NOT EXISTS mifid_catalog.mifid_schema.employees_managers (
        employee_CEN STRING COMMENT 'Employee CEN identifier',
        employee_profession STRING COMMENT 'Short description of the employee profession',
        manager_CEN STRING COMMENT 'Manager CEN identifier',
        manager_updated_datetime TIMESTAMP COMMENT 'Date and time when the manager record was last updated'
    )
    USING DELTA
""")

df_dami = spark.table("cip_catalog.cleansed_dami_owner.employees")
df_parties = spark.table("cip_catalog.cleansed_dwh_owner.parties")
df_professions = spark.table("cip_catalog.cleansed_dwh_owner.person_professions")
df_employees = spark.table("cip_catalog.cleansed_dwh_owner.employees")

df = df_dami.alias("dami").join(df_parties.alias("parties"), "PT_UNIFIED_KEY", "left_outer") \
                .join(df_employees.alias("employees"), ["PT_KEY", "PTORI_KEY"], "inner") \
                .join(df_professions.alias("professions"), "PROF_KEY", "left_outer") \
                .join(df_employees.alias("managers"), F.col("employees.EMP_MANAGER_KEY") == F.col("managers.EMP_KEY"), "left_outer") \
                    .select(
                        F.col("parties.PT_SOURCE_ID").alias("employee_source_id"),
                        F.col("dami.PT_SOURCE_ID").alias("employee_CEN"),
                        F.col("professions.PROF_SHORT_DESC").alias("employee_profession"),
                        F.col("managers.PT_KEY").alias("PT_KEY"),
                        F.col("managers.PTORI_KEY").alias("PTORI_KEY"),
                        F.col("employees.EMP_UPDATED_DATETIME").alias("manager_updated_datetime")
                    ) \
                .join(df_parties, ["PT_KEY", "PTORI_KEY"], "left_outer") \
                .join(df_dami.alias("dami_manager"), "PT_UNIFIED_KEY", "left_outer") \
                    .select(
                        F.col("employee_source_id"),
                        F.col("employee_CEN"),
                        F.col("employee_profession"),
                        F.col("dami_manager.PT_SOURCE_ID").alias("manager_CEN"),
                        F.col("manager_updated_datetime")
                    ) \
                    .withColumn("cnt", F.count("*").over(Window.partitionBy("employee_CEN"))) \
                    .filter(
                        (F.col("cnt") == 1) |
                        (
                            F.substring(F.col("employee_CEN"), -5, 5) ==
                            F.substring(F.col("employee_source_id"), -5, 5)
                        )
                    ) \
                    .drop("cnt") \
                    .withColumn("rn", F.row_number().over(Window.partitionBy("employee_CEN").orderBy(F.col("manager_updated_datetime").desc()))) \
                    .filter(F.col("rn") == 1) \
                    .drop("rn", "employee_source_id", "manager_updated_datetime")


df.write.insertInto("mifid_catalog.mifid_schema.employees_managers", overwrite=True)